In [ ]:
import matplotlib.pyplot as plt
# helper function for data visualization
def visualize(**images):
	"""PLot images in one row."""
	n = len(images)
	plt.figure(figsize=(16, 5))
	for i, (name, image) in enumerate(images.items()):
		plt.subplot(1, n, i + 1)
		plt.xticks([])
		plt.yticks([])
		plt.title(' '.join(name.split('_')).title())
		plt.imshow(image)
	plt.show()

In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision.transforms import ToTensor, Normalize

from glob import glob
from pathlib import Path
from tifffile import imread
import cv2
import numpy as np

In [ ]:
import os
os.cpu_count()

In [ ]:
torch.cuda.empty_cache()

In [ ]:
# setting device on GPU if available, else CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
print()

#Additional Info when using cuda
if device.type == 'cuda':
    print(torch.cuda.get_device_name(0))
    print('Memory Usage:')
    print('Allocated:', round(torch.cuda.memory_allocated(0)/1024**3,1), 'GB')
    print('Cached:   ', round(torch.cuda.memory_reserved(0)/1024**3,1), 'GB')


# CONSTANT DEF

In [ ]:
# Available options are: ['unet', 'unetplusplus', 'manet', 'linknet', 'fpn', 'pspnet', 'deeplabv3', 'deeplabv3plus', 'pan']"

# classes = ['background', 'lepidic', 'acinar', 'micropapillary', 'papillary', 'solid']
classes = ['background', 'solid', 'micropapillary']
class_num = len(classes)
root = r"D:\OneDrive-CMU\Desktop_Dell\JJ\AILCAP\export\512"

arch = 'unetplusplus'
encoder_name = 'resnet34'
lr = 0.001

batch_size = 8
num_workers = 0


# Dataset pipeline

In [ ]:
class CustomDataset(Dataset):
	def __init__(self, root, transform=None, preprocessing=None):
		root = Path(root)
		self.transform = transform
		self.preprocessing = preprocessing

		self.images_path = []
		self.masks_path = []

		scale = ["1.0", "1.5", "2.0", "4.0"]
		for s in scale:
			self.images_path += glob(str(root / f"{s}/images/*"))
			self.masks_path += glob(str(root / f"{s}/masks/*"))

		self.images_path.sort()
		self.masks_path.sort()
	
	
	def get_info(self):
		for mask_path in self.masks_path:
			masks = imread(mask_path)
			print(masks.shape)
			return 


	def __len__(self):
		return len(self.images_path)


	def __getitem__(self, idx):
		img = imread(self.images_path[idx])
		masks = imread(self.masks_path[idx]) # 1 mask contains 5 separated class masks (drop out background)

		if self.preprocessing:
			img = self.preprocessing(img)

		if self.transform:
			transformed = self.transform(image=img, masks=masks)
			img = transformed['image']
			masks = transformed['masks']


		masks = torch.from_numpy(np.asarray(masks)/255).to(torch.int8)

		return img, masks

		# return torch.as_tensor(img), torch.as_tensor(masks)


In [ ]:
from albumentations import Compose, Normalize, HorizontalFlip, RandomCrop, Rotate
from albumentations.pytorch import ToTensorV2

def get_transforms():
    return Compose([
        HorizontalFlip(p=0.5),
		Rotate(limit=180, p=0.5), 
        # RandomCrop(height=256, width=256),
        # Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),  # Converts both images and masks to tensors
    ])

In [ ]:
from segmentation_models_pytorch.encoders import get_preprocessing_fn
preprocess_input = get_preprocessing_fn(encoder_name, pretrained='imagenet')

In [ ]:
# declare dataset
dataset = CustomDataset(root=root, transform=get_transforms(), preprocessing=preprocess_input)

In [ ]:
# sanity check
from random import randint 
r = randint(0, len(dataset)-1)
# r = 65
img, masks = dataset[r]

print("Dataset length: ", len(dataset))
print("Image #", r)
print(f"img shape: {img.shape} | {img.dtype}")
print(f"masks shape: {masks.shape} | {masks.unique()} | {masks.dtype}")

visualize(img=img.permute(1, 2, 0), **{k: v for k, v in zip(classes, masks)})


# Model training

In [ ]:
import segmentation_models_pytorch as smp


model = smp.create_model(
    arch=arch, 
    encoder_name=encoder_name, 
    encoder_weights='imagenet', 
    in_channels=3, 
    classes=class_num 
)
model.to(device)

In [ ]:
import segmentation_models_pytorch.losses as losses
mode = losses.constants.MULTILABEL_MODE

"""
	loss = loss1+loss2+loss3
	loss.backward()
	print(x.grad)
"""

jaccard = losses.JaccardLoss(mode)
dice = losses.DiceLoss(mode)
tversky = losses.TverskyLoss(mode)
focal = losses.FocalLoss(mode)
Lovasz = losses.LovaszLoss(mode)


In [ ]:
from torch import optim

optimizer = optim.Adam(model.parameters(), lr=lr)

In [ ]:
"""
	train : 80%
	val : 10%
	test : 10%
"""
test_portion = 0.1
valid_portion = 0.2 # out of training set

test_len = int(len(dataset) * test_portion)
valid_len = int(len(dataset) * valid_portion)

train_len = len(dataset) - test_len - valid_len

train, valid, test = random_split(dataset=dataset, lengths=[train_len, valid_len, test_len])
print(f"Train: {len(train)}, Validation: {len(valid)}, Test: {len(test)}")

In [ ]:
train_loader = DataLoader(test, batch_size, True, num_workers=num_workers)
valid_loader = DataLoader(valid, batch_size, True, num_workers=num_workers)
test_loader = DataLoader(test, batch_size, True, num_workers=num_workers)

In [ ]:
len(train_loader), len(valid_loader), len(test_loader)

In [ ]:
import segmentation_models_pytorch.metrics.functional as func
from tqdm import tqdm
import gc

from torch.utils.tensorboard import SummaryWriter
writer = SummaryWriter()


epochs = 20
for epoch in tqdm(range(epochs), desc="Epoch"):
	model.train()
	for idx, batch in enumerate(train_loader, start=1):
		images, maskss = batch
		images, maskss = images.to(device).float(), maskss.to(device)

		# [batch_size, channels, height, width]
		print(images.shape, maskss.shape) 

		y_pred = model(images)

		loss = jaccard(y_pred, maskss)
		writer.add_scalar("Loss/train", loss, epoch)

		optimizer.zero_grad()

		loss.backward()
		optimizer.step()
  
		tp, fp, fn, tn = smp.metrics.get_stats(y_pred, maskss, mode='multilabel', threshold=0.5)
		iou_score = smp.metrics.iou_score(tp, fp, fn, tn, reduction="micro")
		f1_score = smp.metrics.f1_score(tp, fp, fn, tn, reduction="micro")
		f2_score = smp.metrics.fbeta_score(tp, fp, fn, tn, beta=2, reduction="micro")
		accuracy = smp.metrics.accuracy(tp, fp, fn, tn, reduction="macro")
		recall = smp.metrics.recall(tp, fp, fn, tn, reduction="micro-imagewise")
		print(f"Batch # {idx}| iou_score={iou_score} f1_score={f1_score} f2_score={f2_score} accuracy={accuracy} recall={recall}")

		del images
		del maskss
		del loss
		del y_pred
		torch.cuda.empty_cache()
		gc.collect

writer.flush()

In [ ]:
MODEL_PATH = Path("models")
MODEL_PATH.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "test.pth"
MODEL_SAVE_PATH = MODEL_PATH / MODEL_NAME

print(MODEL_SAVE_PATH)
# print(f"Saving model to: {MODEL_SAVE_PATH}")
# torch.save(obj=model.state_dict(), # only saving the state_dict() only saves the models learned parameters
#            f=MODEL_SAVE_PATH) 

In [ ]:
loaded_model = smp.create_model(
    arch=arch, 
    encoder_name=encoder_name, 
    encoder_weights='imagenet', 
    in_channels=3, 
    classes=class_num
)

loaded_model.load_state_dict(torch.load(f=MODEL_SAVE_PATH))

In [ ]:
def gray_to_rgb(x):
	expanded_array = np.expand_dims(x, axis=-1)
	x_reshaped = np.repeat(expanded_array, repeats=3, axis=-1)

	# color_map = {
	#         1: [255, 0, 0],   # Red for lepidic
	#         2: [0, 255, 0],   # Green for acinar
	#         3: [0, 0, 255],    # Blue for micropapillary
	#         4: [255, 255, 0],  # Yellow for papillary
	#         5: [255, 0, 255],   # violet for solid
	#     }
 
	color_map = {
			0: [0, 0, 0], # Black for background
			1: [255, 0, 0],   # Red for solid
			2: [0, 255, 0],   # Green for micropapillary
		}

	rgb = np.zeros_like(x_reshaped, dtype=np.uint8)
	for label, color in color_map.items():
			rgb[x_reshaped[..., 0] == label] = color 
	return rgb

In [ ]:
classes_pred = ["p_background", "p_solid", "p_micropapillary"]
model.eval()
for batch in test_loader:
	images, maskss = batch
	images, maskss = images.to(device).float(), maskss.to(device).long()
	print(images.shape)

	with torch.inference_mode():
		y_preds = model(images).cpu()
		arg_max = torch.argmax(y_preds, dim=1)
		# print(arg_max.shape)
		# print(arg_max.unique())
		rgb = gray_to_rgb(arg_max)

		for i in range(batch_size):
			images, maskss = images.cpu(), maskss.cpu()
			visualize(
				img=images[i].permute(1, 2, 0), 
				merged=rgb[i],
				# **{a: b for a, b in zip(classes_pred, y_preds[i])},
				**{k: v for k, v in zip(classes, maskss[i])}
    		)
	break 


In [ ]:
l = np.array([[0, 1, 2]])
expanded_array = np.expand_dims(l, axis=-1)

# Duplicate along the new dimension to get shape (8, 512, 512, 3)
# This can also be achieved with np.tile if you want a specific pattern
output_array = np.repeat(expanded_array, repeats=3, axis=-1)

output_array